# IC-1019 — Case study: approve this card?

**ML & FinTech · 115-1 · 20261019 · Session 3 · work in pairs · 40 minutes, then discussion**

| Step | Min | What you do |
|---|---|---|
| 0 | 3 | Sit with a partner. Fill in both IDs; your desk gets its own 3,000 applicants |
| 1 | 10 | Default probability **by hand** from the fitted equation, then check with code |
| 2 | 12 | Fit the model on your own desk; read the coefficient the right way |
| 3 | 10 | Decide two applications; compare with your partner |
| 4 | 5 | Hand in **one sheet of paper per pair** |

You may use any AI tool for the code. The graded part is the paper your pair hands in, and what
you say when called on. Your numbers come from your 學號; your partner's differ.

---

## The case

You are on the credit desk. A junior analyst sends this memo about the bank's default model
(a logistic regression on credit limit and months late on the last payment):

> *"The coefficient on months late is about **0.69**, so **each month late adds 69 percentage points
> to the probability of default**. Applicant A (limit NT$50,000, 2 months late) is therefore
> near-certain to default: **reject**. Applicant B (limit NT$300,000, paid on time) is safe: **approve**."*

Your manager asks: **"Is the memo right? What do we tell A and B?"**

## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/main/slides/"
        "default%20of%20credit%20card%20clients.csv")
CAND = ["default of credit card clients.csv", "../slides/default of credit card clients.csv",
        "../00-course-info/slides/default of credit card clients.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
credit = pd.read_csv(path).rename(columns={"X1":"LIMIT","X5":"AGE","X6":"PAY_0","X7":"PAY_2",
                                           "X12":"BILL_1","X18":"PAY_AMT_1","Y":"default"})
credit["LIMIT"] = credit["LIMIT"] / 10000          # NT$ ten-thousands
credit["BILL_1"] = credit["BILL_1"] / 10000
credit["PAY_AMT_1"] = credit["PAY_AMT_1"] / 10000
credit = credit[credit["PAY_0"] <= 8]                # one record has PAY_0 = 3032: a data error (IC-02). Dropped.
# 30,000 Taiwanese credit-card clients (2005). default = 1 if the client defaulted next month.
# PAY_0: months late on the most recent payment (-1 = paid on time, 0 = revolving, 1..9 months late).

def my_sample(student_id, n=3000):
    """Your desk's 3,000 applicants. The same 學號 always gives the same applicants."""
    seed = int(hashlib.sha256(f"{int(student_id)}-credit".encode()).hexdigest(), 16) % 2**32
    rng = np.random.default_rng(seed)
    return credit.iloc[np.sort(rng.choice(len(credit), n, replace=False))].copy()

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
desk = my_sample(STUDENT_ID)
print(f"Desk of {STUDENT_ID}: {len(desk)} applicants, {desk['default'].mean()*100:.1f}% defaulted")

---
# Step 1 — The equation by hand (10 min)

The model fitted on all 30,000 clients (LIMIT in NT$10,000s, PAY_0 = months late):

log-odds = **-1.122** + (**-0.018**) × LIMIT + (**0.694**) × PAY_0, and p = e^(log-odds) / (1 + e^(log-odds)).

**Q1.** By hand, to two decimals:

- Applicant A (LIMIT = 5, PAY_0 = 2): log-odds = ____ , p(default) = ____
- Applicant B (LIMIT = 30, PAY_0 = 0): log-odds = ____ , p(default) = ____

**Q2.** The memo says one month late "adds 69 percentage points". Compute p for A with PAY_0 = 3
instead of 2. How much did p actually rise? What does e^0.69 ≈ 2.00 describe instead?

> _replace this line_

---
# Step 2 — Your own desk (12 min)

Split your desk into a fitting part and a test part (given), then fit a logistic regression of
`default` on `LIMIT` and `PAY_0` with `sklearn.linear_model.LogisticRegression(max_iter=2000)`
on `train`. Print the intercept and the two coefficients.

In [ ]:
from sklearn.model_selection import train_test_split
seed = int(hashlib.sha256(f"{int(STUDENT_ID)}-split".encode()).hexdigest(), 16) % 2**32
train, test = train_test_split(desk, test_size=0.4, random_state=seed, stratify=desk["default"])
print(len(train), "to fit,", len(test), "to test")

In [ ]:
# 2: fit on train; print intercept and coefficients



**Q3.** My coefficients: LIMIT ____ , PAY_0 ____ . Odds ratio for one extra month late, e^(coef) = ____ .

**Q4.** Is the memo's reading of the coefficient right or wrong? One sentence, using your own number.

> _replace this line_

---
# Step 3 — Decide, and compare (10 min)

With your fitted model, compute p(default) for A (LIMIT 5, PAY_0 2) and B (LIMIT 30, PAY_0 0).
Decide each at the bank's cutoff of 0.5.

In [ ]:
# 3: predict_proba for A and B



**Q5.** My model: p(A) = ____ → ____ ; p(B) = ____ → ____ .

**Q6.** Swap with your partner.

| | coef PAY_0 | p(A) | decision on A |
|---|---|---|---|
| me | | | |
| my partner | | | |

Same model, different desks: why do the numbers differ, and does the decision differ? Is "near-certain" the right word for A?

> _replace this line_

---
# Step 4 — Hand in, on paper (5 min)

**One sheet per pair.** Write:

1. Both names and both 學號.
2. For **each** of you: the coefficient on PAY_0 and p(A).
3. **Two sentences, agreed between you**, answering the client's question and the trap.

Give the sheet to Prof. Teng or Claire before you leave. **The sheet is the graded part.**

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1019/IC-1019.ipynb`
in your own repository before **12:10**.